# Casualty Generator
Replicates `src/helpers/casualty_generator.py` as an interactive notebook.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Callable, Tuple, Union
import random

import numpy as np
import pandas as pd
import salabim as sim
import yaml

In [ ]:

# ── config.yaml ──────────────────────────────────────────────────────────────
_CONFIG_YAML = """
# Spawn locations
ccp_ids:
  - ccp1
  - ccp2
role1_ids:
  - r1_bde1
  - r1_bde2

# Global Personnel At Risk
par_total: 18800

# Wave scheduling
n_days: 3
waves_per_day: 3
wave_duration_min: 30

# Casualty rates per soldier per day (Meisner et al. 2025)
rate_bcas: 0.023    # Battle Casualties (intense scenario)
rate_dnbi: 0.0013   # Disease and Non-Battle Injury

# Random seed
seed: 42
"""

# ── BCAS diagnosis mix ────────────────────────────────────────────────────────
# Of all BCAS: 20% KIA, 10% MIA, 70% WIA
# Of WIA: 15% T1, 20% T2, 65% T3  →  T1=0.70*0.15=0.105, T2=0.140, T3=0.455
bcas_mix_df = pd.DataFrame({
    "triage":                ["T1_BCAS",    "T2_BCAS",    "T3_BCAS",    "KIA",   "MIA"  ],
    "frequency":             [ 0.105,        0.140,         0.455,        0.200,   0.100  ],
    "lifetime_distribution": ["triangular",  "triangular",  "triangular", "triangular", "triangular"],
    "lifetime_low":          [ 40,           1200,          10000,        1,       1      ],
    "lifetime_mode":         [ 60,           1440,          11000,        1,       1      ],
    "lifetime_high":         [ 80,           1680,          12000,        2,       2      ],
})

# ── DNBI diagnosis mix ────────────────────────────────────────────────────────
# All DNBI enter the care pathway directly at Role 1 (no KIA/MIA)
dnbi_mix_df = pd.DataFrame({
    "triage":                ["T1_DNBI",    "T2_DNBI",    "T3_DNBI"  ],
    "frequency":             [ 0.15,         0.20,         0.65       ],
    "lifetime_distribution": ["triangular",  "triangular",  "triangular"],
    "lifetime_low":          [ 40,           1200,          10000      ],
    "lifetime_mode":         [ 60,           1440,          11000      ],
    "lifetime_high":         [ 80,           1680,          12000      ],
})

# ── prescriptions ─────────────────────────────────────────────────────────────
# Surgery fractions per Meisner et al. (2025):
#   T1: 95% DCS  |  T2: 80% DCS  |  T3_BCAS: 80% DCS  |  T3_DNBI: 20% DCS
prescriptions_df = pd.DataFrame({
    "label": [
        "T1_BCAS", "T1_BCAS",
        "T2_BCAS", "T2_BCAS",
        "T3_BCAS", "T3_BCAS",
        "T1_DNBI", "T1_DNBI",
        "T2_DNBI", "T2_DNBI",
        "T3_DNBI", "T3_DNBI",
        "KIA",
        "MIA",
    ],
    "frequency": [
        0.95, 0.05,
        0.80, 0.20,
        0.80, 0.20,
        0.95, 0.05,
        0.80, 0.20,
        0.20, 0.80,
        1.00,
        1.00,
    ],
    "prescription": [
        ["DCR", "DCS"], ["DCR"],
        ["DCR", "DCS"], ["DCR"],
        ["PEC", "DCS"], ["PEC"],
        ["DCR", "DCS"], ["DCR"],
        ["DCR", "DCS"], ["DCR"],
        ["PEC", "DCS"], ["PEC"],
        [],
        [],
    ],
})

# ── parse params ──────────────────────────────────────────────────────────────
params = yaml.safe_load(_CONFIG_YAML)


In [ ]:
# ============================================================
# DATA MODEL
# ============================================================

@dataclass
class Wave:
    wave_id: str
    t_start: float          # minutes from sim start
    t_end: float            # minutes from sim start
    facility_id: str
    n_casualties: int = 0
    df: Optional[pd.DataFrame] = None  # at least: t_create, spawn_facility_id


WaveDict = Dict[str, List[Wave]]  # facility_id -> list[Wave]

In [ ]:
# ============================================================
# HELPERS
# ============================================================

def _normalize_probs(p: np.ndarray) -> np.ndarray:
    s = float(p.sum())
    if s <= 0:
        raise ValueError("Probabilities must sum to > 0")
    return p / s

In [ ]:
# ============================================================
# LIFETIME FACTORY BUILDER
# ============================================================

def _lifetime_df_to_factories(diagnosis_mix_df: pd.DataFrame) -> Dict[str, Callable[[], object]]:
    """Convert diagnosis_mix dataframe to lifetime factory functions."""
    factories = {}

    for _, row in diagnosis_mix_df.iterrows():
        triage = row["triage"]
        dist_type = row["lifetime_distribution"].lower()

        if dist_type == "uniform":
            low, high = row["lifetime_low"], row["lifetime_high"]
            factories[triage] = lambda l=low, h=high: sim.Uniform(l, h)
        elif dist_type == "triangular":
            low = row["lifetime_low"]
            high = row["lifetime_high"]
            mode = row.get("lifetime_mode", (low + high) / 2)
            if pd.isna(mode):
                mode = (low + high) / 2
            factories[triage] = lambda l=low, m=mode, h=high: sim.Triangular(l, h, m)
        elif dist_type == "exponential":
            mean = row["lifetime_mean"]
            factories[triage] = lambda m=mean: sim.Exponential(m)
        elif dist_type == "normal":
            mean, std = row["lifetime_mean"], row["lifetime_std"]
            factories[triage] = lambda m=mean, s=std: sim.Normal(m, s)
        else:
            raise ValueError(f"Unknown distribution type: {dist_type}")

    return factories

In [ ]:
# ============================================================
# STEP A) WAVE PLACER
#   - creates wave_dict with Wave objects (no casualties yet)
#   - random placement per day per facility
#   - NO overlap by construction using random slot selection
# ============================================================

def place_waves_randomly_per_day(
    facility_ids: List[str],
    n_days: int,
    waves_per_day: int,
    wave_duration_min: int,
    seed: int = 123,
) -> WaveDict:
    """
    Places 'waves_per_day' non-overlapping waves of duration 'wave_duration_min'
    randomly within each day for each facility.

    Slot method:
      day split into S = floor(1440 / wave_duration_min) slots.
      For each (facility, day) sample 'waves_per_day' distinct slots.
    """
    rng = random.Random(seed)
    minutes_per_day = 24 * 60

    if wave_duration_min <= 0:
        raise ValueError("wave_duration_min must be > 0")
    if n_days <= 0:
        raise ValueError("n_days must be > 0")
    if waves_per_day <= 0:
        raise ValueError("waves_per_day must be > 0")
    if not facility_ids:
        raise ValueError("facility_ids must be a non-empty list")

    slots_per_day = minutes_per_day // wave_duration_min
    if waves_per_day > slots_per_day:
        raise ValueError(
            f"waves_per_day={waves_per_day} too large for wave_duration_min={wave_duration_min}. "
            f"Max is {slots_per_day}."
        )

    wave_dict: WaveDict = {fid: [] for fid in facility_ids}
    wave_counter = 0
    slot_indices = list(range(slots_per_day))

    for day in range(n_days):
        day_start = day * minutes_per_day
        for fid in facility_ids:
            chosen_slots = rng.sample(slot_indices, k=waves_per_day)
            chosen_slots.sort()
            for s in chosen_slots:
                t_start = day_start + s * wave_duration_min
                t_end = t_start + wave_duration_min
                wave_id = f"w_{wave_counter:06d}"
                wave_counter += 1
                wave_dict[fid].append(Wave(
                    wave_id=wave_id,
                    t_start=float(t_start),
                    t_end=float(t_end),
                    facility_id=fid,
                ))

    return wave_dict

In [ ]:

# ============================================================
# STEP B) WAVE CASUALTY ALLOCATOR
#
# For each day:
#   N_day_total ~ Poisson(par_total * rate_per_day)
# Distribute across facilities (weighted multinomial),
# then across that facility's waves (random multinomial).
# ============================================================

def allocate_casualties_to_waves(
    wave_dict: WaveDict,
    par_total: int,
    rate_per_day: float,
    facility_weights: Optional[Dict[str, float]] = None,
    seed: int = 123,
) -> None:
    """
    Mutates wave_dict in-place by setting wave.n_casualties.

    - par_total:        global PAR (e.g., 18800)
    - rate_per_day:     casualties per soldier per day
    - facility_weights: optional weights per facility_id; None = equal weights
    """
    if par_total < 0:
        raise ValueError("par_total must be >= 0")
    if rate_per_day < 0:
        raise ValueError("rate_per_day must be >= 0")

    rng = np.random.default_rng(seed)
    minutes_per_day = 24 * 60
    facility_ids = list(wave_dict.keys())
    if not facility_ids:
        return

    if facility_weights is None:
        weights = np.ones(len(facility_ids), dtype=float)
    else:
        weights = np.array(
            [float(facility_weights.get(fid, 0.0)) for fid in facility_ids], dtype=float
        )
        if (weights < 0).any():
            raise ValueError("facility_weights must be non-negative")
        if float(weights.sum()) <= 0:
            raise ValueError("facility_weights sum must be > 0 (or pass None for equal split)")

    weights = _normalize_probs(weights)

    all_days = set()
    for fid in facility_ids:
        for w in wave_dict[fid]:
            all_days.add(int(w.t_start // minutes_per_day))
    if not all_days:
        return
    n_days = max(all_days) + 1

    for day_idx in range(n_days):
        lam_day = par_total * rate_per_day
        n_day_total = int(rng.poisson(lam=lam_day))

        if n_day_total <= 0:
            for fid in facility_ids:
                for w in wave_dict[fid]:
                    if int(w.t_start // minutes_per_day) == day_idx:
                        w.n_casualties = 0
            continue

        # Split across facilities
        n_by_facility = rng.multinomial(n_day_total, weights)

        for fid, n_fac_day in zip(facility_ids, n_by_facility):
            day_waves = [
                w for w in wave_dict[fid]
                if int(w.t_start // minutes_per_day) == day_idx
            ]
            if not day_waves:
                continue

            day_waves.sort(key=lambda ww: ww.t_start)
            k = len(day_waves)

            # Random split across waves (multinomial, equal weights)
            wave_weights = np.ones(k, dtype=float) / k
            counts = rng.multinomial(int(n_fac_day), wave_weights)
            for w, c in zip(day_waves, counts):
                w.n_casualties = int(c)


In [ ]:
# ============================================================
# STEP C) WAVE FILLER
#   - creates wave.df with ONLY: t_create, spawn_facility_id
#   - inside each wave: conditional on n, times ~ Uniform(t_start, t_end)
# ============================================================

def fill_wave_dataframes(wave_dict: WaveDict, seed: int = 123) -> None:
    rng = np.random.default_rng(seed)

    for fid, waves in wave_dict.items():
        for w in waves:
            n = int(w.n_casualties)
            if n <= 0:
                w.df = pd.DataFrame(columns=["t_create", "spawn_facility_id"])
                continue
            t = rng.uniform(low=w.t_start, high=w.t_end, size=n)
            w.df = (
                pd.DataFrame({"t_create": t.astype(float), "spawn_facility_id": [fid] * n})
                .sort_values("t_create")
                .reset_index(drop=True)
            )

In [ ]:
# ============================================================
# STEP D) CASUALTY POPULATOR
#   Adds: triage, prescription, lifetime_dist
# ============================================================

def populate_patient_attributes(
    wave_dict: WaveDict,
    diagnosis_mix_df: pd.DataFrame,
    rx_df: pd.DataFrame,
    lifetime_factory_by_triage: Optional[Dict[str, Callable[[], object]]] = None,
    seed: int = 123,
) -> None:
    rng = np.random.default_rng(seed)

    diag_labels = diagnosis_mix_df["triage"].astype(str).to_numpy()
    diag_p = _normalize_probs(diagnosis_mix_df["frequency"].astype(float).to_numpy())

    if lifetime_factory_by_triage is None:
        lifetime_factory_by_triage = {}
    default_lifetime_factory = lambda: sim.Uniform(60, 120)

    rx_df = rx_df.copy()
    rx_df["label"] = rx_df["label"].astype(str)

    rx_by_label: Dict[str, Tuple[np.ndarray, List[List[str]]]] = {}
    for label, g in rx_df.groupby("label"):
        presc_list = [list(x) for x in g["prescription"].tolist()]
        p = _normalize_probs(g["frequency"].astype(float).to_numpy())
        rx_by_label[label] = (p, presc_list)

    global_presc_list = [list(x) for x in rx_df["prescription"].tolist()]
    global_p = _normalize_probs(rx_df["frequency"].astype(float).to_numpy())

    for _, waves in wave_dict.items():
        for w in waves:
            if w.df is None or len(w.df) == 0:
                if w.df is None:
                    w.df = pd.DataFrame(columns=["t_create", "spawn_facility_id"])
                w.df["triage"] = pd.Series(dtype=str)
                w.df["prescription"] = pd.Series(dtype=object)
                w.df["lifetime_dist"] = pd.Series(dtype=object)
                continue

            n = len(w.df)
            triage = rng.choice(diag_labels, size=n, p=diag_p).astype(str)

            prescriptions: List[List[str]] = []
            lifetime_dists: List[object] = []

            for tr in triage:
                if tr in rx_by_label:
                    p, presc_list = rx_by_label[tr]
                    idx = int(rng.choice(np.arange(len(presc_list)), p=p))
                    prescriptions.append(list(presc_list[idx]))
                else:
                    idx = int(rng.choice(np.arange(len(global_presc_list)), p=global_p))
                    prescriptions.append(list(global_presc_list[idx]))

                lf_factory = lifetime_factory_by_triage.get(tr, default_lifetime_factory)
                lifetime_dists.append(lf_factory())

            w.df = w.df.copy()
            w.df["triage"] = triage
            w.df["prescription"] = prescriptions
            w.df["lifetime_dist"] = lifetime_dists

In [ ]:

# ============================================================
# STEP E) MERGE WAVES -> FINAL casualties_df
#   Accepts one or more WaveDicts (e.g. BCAS + DNBI separately)
# ============================================================

def merge_waves_to_casualties_df(*wave_dicts: WaveDict) -> pd.DataFrame:
    frames = []
    for wave_dict in wave_dicts:
        for waves in wave_dict.values():
            for w in waves:
                if w.df is not None and len(w.df) > 0:
                    frames.append(w.df)

    if not frames:
        return pd.DataFrame(
            columns=["t_create", "spawn_facility_id", "triage", "prescription", "lifetime_dist"]
        )

    out = pd.concat(frames, ignore_index=True)
    return out.sort_values("t_create").reset_index(drop=True)


In [ ]:
# ============================================================
# SAVE HELPER
# ============================================================

def save_casualties_df(
    casualties_df: pd.DataFrame,
    output_path: Union[str, Path],
) -> Path:
    """Save casualties dataframe to XLSX, extracting lifetime bounds."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    save_df = casualties_df.copy()

    def get_dist_type(d):
        cls_name = type(d).__name__.lower()
        if "triangular" in cls_name:  return "triangular"
        elif "uniform" in cls_name:   return "uniform"
        elif "exponential" in cls_name: return "exponential"
        elif "normal" in cls_name:    return "normal"
        return "uniform"

    def get_low(d):
        for attr in ["_lowerbound", "low", "a", "min"]:
            if hasattr(d, attr): return getattr(d, attr)
        return None

    def get_high(d):
        for attr in ["_upperbound", "high", "b", "max"]:
            if hasattr(d, attr): return getattr(d, attr)
        return None

    def get_mode(d):
        for attr in ["_mode", "mode", "c"]:
            if hasattr(d, attr): return getattr(d, attr)
        return None

    save_df["lifetime_dist_type"] = save_df["lifetime_dist"].apply(get_dist_type)
    save_df["lifetime_low"]       = save_df["lifetime_dist"].apply(get_low)
    save_df["lifetime_high"]      = save_df["lifetime_dist"].apply(get_high)
    save_df["lifetime_mode"]      = save_df["lifetime_dist"].apply(get_mode)

    save_cols = [c for c in save_df.columns if c != "lifetime_dist"]
    save_df[save_cols].to_excel(output_path, index=False)
    print(f"Saved to: {output_path}")
    return output_path

In [ ]:

# ============================================================
# ORCHESTRATOR
#   Runs the full pipeline for BCAS (CCPs) and DNBI (Role 1s)
#   separately, then merges into one casualties_df.
# ============================================================

def generate_casualties_df_from_waves(
    ccp_ids: List[str],
    role1_ids: List[str],
    par_total: int,
    n_days: int,
    waves_per_day: int,
    wave_duration_min: int,
    rate_bcas: float,
    rate_dnbi: float,
    bcas_mix_df: pd.DataFrame,
    dnbi_mix_df: pd.DataFrame,
    rx_df: pd.DataFrame,
    facility_weights_ccp: Optional[Dict[str, float]] = None,
    facility_weights_r1:  Optional[Dict[str, float]] = None,
    lifetime_factory_by_triage: Optional[Dict[str, Callable[[], object]]] = None,
    seed: int = 123,
) -> Tuple[pd.DataFrame, WaveDict, WaveDict]:
    """
    Returns (casualties_df, wave_dict_bcas, wave_dict_dnbi).

    casualties_df contains ALL generated entities including KIA and MIA.
    Filter with  simulation_input_df = casualties_df[~casualties_df['triage'].isin(['KIA','MIA'])]
    before passing to the simulator.
    """

    # ── BCAS: battle casualties spawn at CCPs ────────────────────────────────
    wave_dict_bcas = place_waves_randomly_per_day(
        facility_ids=ccp_ids,
        n_days=n_days,
        waves_per_day=waves_per_day,
        wave_duration_min=wave_duration_min,
        seed=seed,
    )
    allocate_casualties_to_waves(
        wave_dict=wave_dict_bcas,
        par_total=par_total,
        rate_per_day=rate_bcas,
        facility_weights=facility_weights_ccp,
        seed=seed + 1,
    )
    fill_wave_dataframes(wave_dict=wave_dict_bcas, seed=seed + 2)
    populate_patient_attributes(
        wave_dict=wave_dict_bcas,
        diagnosis_mix_df=bcas_mix_df,
        rx_df=rx_df,
        lifetime_factory_by_triage=lifetime_factory_by_triage,
        seed=seed + 3,
    )

    # ── DNBI: disease/non-battle casualties spawn at Role 1 MTFs ────────────
    wave_dict_dnbi = place_waves_randomly_per_day(
        facility_ids=role1_ids,
        n_days=n_days,
        waves_per_day=waves_per_day,
        wave_duration_min=wave_duration_min,
        seed=seed + 10,
    )
    allocate_casualties_to_waves(
        wave_dict=wave_dict_dnbi,
        par_total=par_total,
        rate_per_day=rate_dnbi,
        facility_weights=facility_weights_r1,
        seed=seed + 11,
    )
    fill_wave_dataframes(wave_dict=wave_dict_dnbi, seed=seed + 12)
    populate_patient_attributes(
        wave_dict=wave_dict_dnbi,
        diagnosis_mix_df=dnbi_mix_df,
        rx_df=rx_df,
        lifetime_factory_by_triage=lifetime_factory_by_triage,
        seed=seed + 13,
    )

    # ── Merge ────────────────────────────────────────────────────────────────
    casualties_df = merge_waves_to_casualties_df(wave_dict_bcas, wave_dict_dnbi)
    return casualties_df, wave_dict_bcas, wave_dict_dnbi


In [ ]:

# ============================================================
# RUN
# ============================================================

SAVE_OUTPUT = True
OUTPUT_PATH = "data/single_run/input/casualty_dataframes/casualties.xlsx"

# Build lifetime factories from both mixes
all_factories = {
    **_lifetime_df_to_factories(bcas_mix_df),
    **_lifetime_df_to_factories(dnbi_mix_df),
}

casualties_df, wave_dict_bcas, wave_dict_dnbi = generate_casualties_df_from_waves(
    ccp_ids=params["ccp_ids"],
    role1_ids=params["role1_ids"],
    par_total=params["par_total"],
    n_days=params["n_days"],
    waves_per_day=params["waves_per_day"],
    wave_duration_min=params["wave_duration_min"],
    rate_bcas=params["rate_bcas"],
    rate_dnbi=params["rate_dnbi"],
    bcas_mix_df=bcas_mix_df,
    dnbi_mix_df=dnbi_mix_df,
    rx_df=prescriptions_df,
    lifetime_factory_by_triage=all_factories,
    seed=params["seed"],
)

# ── Filter out KIA/MIA before passing to simulation ──────────────────────────
simulation_input_df = casualties_df[
    ~casualties_df["triage"].isin(["KIA", "MIA"])
].copy().reset_index(drop=True)

if SAVE_OUTPUT:
    save_casualties_df(simulation_input_df, OUTPUT_PATH)

# ── Summary ───────────────────────────────────────────────────────────────────
print("=== Triage breakdown (all generated) ===")
print(casualties_df["triage"].value_counts().sort_index().to_string())
print(f"\nTotal generated:       {len(casualties_df)}")
print(f"  of which KIA:        {(casualties_df['triage'] == 'KIA').sum()}")
print(f"  of which MIA:        {(casualties_df['triage'] == 'MIA').sum()}")
print(f"\nSimulation input:      {len(simulation_input_df)}")
print(f"\nSpawn locations:")
print(simulation_input_df["spawn_facility_id"].value_counts().sort_index().to_string())
print(f"\nFirst 10 rows of simulation_input_df:")
print(simulation_input_df.head(10))


In [ ]:

# ============================================================
# VISUALIZATION — Casualties per wave slot
#   Pick any facility from ccp_ids (BCAS) or role1_ids (DNBI)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── SELECT ────────────────────────────────────────────────────────────────────
PLOT_FACILITY = "ccp1"   # any value from ccp_ids or role1_ids
PLOT_DAY      = 0        # 0-indexed
# ─────────────────────────────────────────────────────────────────────────────

# Route to the correct wave dict
all_wave_dicts = {**wave_dict_bcas, **wave_dict_dnbi}
if PLOT_FACILITY not in all_wave_dicts:
    raise ValueError(
        f"'{PLOT_FACILITY}' not found. "
        f"Available: {list(all_wave_dicts.keys())}"
    )

wave_dur        = params["wave_duration_min"]
minutes_per_day = 24 * 60
slots_per_day   = minutes_per_day // wave_dur
day_start       = PLOT_DAY * minutes_per_day

slot_counts = np.zeros(slots_per_day, dtype=int)
for w in all_wave_dicts[PLOT_FACILITY]:
    if int(w.t_start // minutes_per_day) == PLOT_DAY:
        slot_idx = int((w.t_start - day_start) // wave_dur)
        slot_counts[slot_idx] += w.n_casualties

fig, ax = plt.subplots(figsize=(14, 4))
colors = ["steelblue" if c > 0 else "lightgrey" for c in slot_counts]
ax.bar(range(slots_per_day), slot_counts, color=colors, edgecolor="white", linewidth=0.5)
for i, c in enumerate(slot_counts):
    if c > 0:
        ax.text(i, c + 0.3, str(c), ha="center", va="bottom", fontsize=8, fontweight="bold")

ax.set_xlabel(f"Slot index  (each slot = {wave_dur} min)")
ax.set_ylabel("Casualties")
ax.set_title(f"Casualties per {wave_dur}-min slot — facility: {PLOT_FACILITY}, day {PLOT_DAY}")
ax.set_xlim(-0.5, slots_per_day - 0.5)
ax.yaxis.get_major_locator().set_params(integer=True)

ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
tick_pos = list(range(0, slots_per_day, max(1, slots_per_day // 12)))
ax2.set_xticks(tick_pos)
ax2.set_xticklabels([f"{(t * wave_dur) // 60:02d}:00" for t in tick_pos], fontsize=8)
ax2.set_xlabel("Time of day")

ax.legend(handles=[
    mpatches.Patch(color="steelblue", label="Wave slots"),
    mpatches.Patch(color="lightgrey", label="Empty slots"),
], loc="upper right")

plt.tight_layout()
plt.show()
print(f"Total: {slot_counts.sum()} | Active slots: {(slot_counts > 0).sum()} / {slots_per_day}")
